In [ ]:
!pip install torch torchvision tensorboard

In [ ]:
!nvidia-smi

Wed Apr 15 23:48:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# dataset
transform = transforms.Compose([transforms.ToTensor()])

dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# model
def create_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 2048),
        nn.ReLU(),
        nn.Linear(2048, 1024),
        nn.ReLU(),
        nn.Linear(1024, 10)
    ).to(device)

# FP32 training
model_fp32 = create_model()
optimizer_fp32 = optim.Adam(model_fp32.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

start = time.time()

for step, (x,y) in enumerate(loader):

    x = x.to(device)
    y = y.to(device)

    pred = model_fp32(x)
    loss = loss_fn(pred, y)

    optimizer_fp32.zero_grad()
    loss.backward()
    optimizer_fp32.step()

    if step > 100:
        break

fp32_time = time.time() - start

print("FP32 time:", fp32_time)

# Mixed precision training
model_amp = create_model()
optimizer_amp = optim.Adam(model_amp.parameters(), lr=1e-3)

scaler = torch.cuda.amp.GradScaler()

start = time.time()

for step, (x,y) in enumerate(loader):

    x = x.to(device)
    y = y.to(device)

    with torch.cuda.amp.autocast():

        pred = model_amp(x)
        loss = loss_fn(pred, y)

    optimizer_amp.zero_grad()

    scaler.scale(loss).backward()

    scaler.step(optimizer_amp)

    scaler.update()

    if step > 100:
        break

amp_time = time.time() - start

print("AMP time:", amp_time)

print("Speed improvement:", fp32_time / amp_time)

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 170kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.16MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.0MB/s]


FP32 time: 5.6078596115112305


/tmp/ipykernel_748/286958413.py:69: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_748/286958413.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


AMP time: 5.207643508911133
Speed improvement: 1.0768516704177737


In [2]:
!nvidia-smi

Fri Apr 17 18:14:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             26W /   70W |     327MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os
import time
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.profiler import profile, ProfilerActivity, schedule

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Clean old logs
for d in ["./log_fp32", "./log_amp"]:
    if os.path.exists(d):
        shutil.rmtree(d)

# Dataset
transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

def create_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 2048),
        nn.ReLU(),
        nn.Linear(2048, 1024),
        nn.ReLU(),
        nn.Linear(1024, 10)
    ).to(device)

loss_fn = nn.CrossEntropyLoss()

def run_profile(run_name="fp32", use_amp=False):
    model = create_model()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # Current recommended API
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    trace_dir = f"./log_{run_name}"
    os.makedirs(trace_dir, exist_ok=True)

    prof_schedule = schedule(wait=1, warmup=1, active=3, repeat=1)

    start = time.time()

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        schedule=prof_schedule,
        record_shapes=True,
        profile_memory=True,
        with_stack=False
    ) as prof:

        for step, (x, y) in enumerate(loader):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    pred = model(x)
                    loss = loss_fn(pred, y)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                pred = model(x)
                loss = loss_fn(pred, y)
                loss.backward()
                optimizer.step()

            prof.step()

            if step >= 20:
                break

    elapsed = time.time() - start

    # Export a Chrome/Perfetto-compatible trace
    trace_file = os.path.join(trace_dir, f"{run_name}.pt.trace.json")
    prof.export_chrome_trace(trace_file)

    sort_key = "cuda_time_total" if torch.cuda.is_available() else "cpu_time_total"
    print(f"\n===== {run_name.upper()} RESULTS =====")
    print("Elapsed time:", round(elapsed, 3), "sec")
    print(prof.key_averages().table(sort_by=sort_key, row_limit=15))
    print("Trace file:", trace_file)

    return elapsed, trace_file

fp32_time, fp32_trace = run_profile("fp32", use_amp=False)
amp_time, amp_trace = run_profile("amp", use_amp=True)

print("\n===== COMPARISON =====")
print("FP32 time:", round(fp32_time, 3), "sec")
print("AMP time :", round(amp_time, 3), "sec")
print("Speedup  :", round(fp32_time / amp_time, 3), "x" if amp_time > 0 else "N/A")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(



===== FP32 RESULTS =====
Elapsed time: 1.743 sec
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      13.916ms        78.93%      13.916ms       4.639ms     

In [4]:
!find ./log_fp32 -type f
!find ./log_amp -type f

./log_fp32/fp32.pt.trace.json
./log_amp/amp.pt.trace.json


In [5]:
from google.colab import files

files.download('/content/log_fp32/fp32.pt.trace.json')
files.download('/content/log_amp/amp.pt.trace.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>